# Steps 7–8, Approach 3: TopicGPT and LLooM, per WO partition

Interactive version of the `llm_topics` stage of `run_pipeline.py`. It runs the AI part of the three-approach comparison **separately for each WO range**, because the range decides what free text an ECR has:

| Partition | What the LLM sees |
|---|---|
| `title_only` (WO 4.1M–4.8M) | Title, cost comment, Driver, Change type |
| `full_text` (WO 7.0M+) | Title, description, reason |

Each partition gets its own codebook (built from the generation sample), then its held-out evaluation ECRs are labeled. Merged topics are renamed so each name fits everything merged into it, every topic has include and exclude rules, and each ECR gets at most 2 "why" labels.

**Before you start:** run `python run_pipeline.py` once (it writes the samples to `outputs/05_topics/<partition>/approach3_llm_inputs/`) and fill in `.env` from `.env.example`. Every request goes through `ecr_topics/llm_caller.py`. Responses are cached in `outputs/05_topics/llm_cache/`, so an interrupted run resumes for free. Results go to `outputs/05_topics/<partition>/approach3_llm/`.

In [ ]:
import sys
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from ecr_topics import llm_topics as T
from ecr_topics.llm_caller import available_models, default_model

TOPICS_OUT = ROOT / "outputs" / "05_topics"
models = available_models()
model_selector = widgets.Dropdown(options=models, value=default_model() if default_model() in models else models[0],
                                  description="Model:", layout=widgets.Layout(width="70%"))
display(model_selector)

## 1. Estimate (sends nothing)

In [ ]:
est = T.run_all(model_selector.value, dry_run=True, topics_out=TOPICS_OUT)
print(est["report"])

## 2. Run
Set `RUN = True` after checking the estimate. Lower `MAX_WORKERS` to 2 if the endpoint returns rate-limit errors.

In [ ]:
RUN = False
MAX_WORKERS = 4
if RUN:
    info = T.run_all(model_selector.value, dry_run=False, max_workers=MAX_WORKERS, topics_out=TOPICS_OUT)
    print(info["report"])
else:
    print("RUN is False: nothing sent.")

## 3. Quick look
Then run `python run_pipeline.py --only topics --with-llm` (cached, so free) to rebuild the comparison and report.

In [ ]:
import json

import pandas as pd

for part in T.PARTITIONS:
    out = TOPICS_OUT / part / "approach3_llm"
    book = json.loads((out / "topicgpt_codebook.json").read_text())
    tg = pd.read_csv(out / "topicgpt_assignments.csv", dtype={"WO_NO": str}).fillna("")
    lm = pd.read_csv(out / "lloom_scores.csv", dtype={"WO_NO": str}).fillna("")
    print(f"== {part}: {len(book['why'])} WHY topics, {len(book['what'])} WHAT topics | "
          f"ECRs with a WHY label: TopicGPT {(tg.why_topics != '').mean():.0%}, LLooM {(lm.concepts != '').mean():.0%}")
    display(pd.DataFrame(book["why"])[["name", "given_name", "definition", "include_rule", "exclude_rule"]])